<a href="https://colab.research.google.com/github/Rei5ende/pytorch-bc-study/blob/main/04_behavior_cloning/bc_cartpole.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#-------------
#데이터 준비
#-------------

In [1]:
# -----------------------------------------------
# 1. 라이브러리 임포트
# -----------------------------------------------

# gymnasium: 강화학습 환경 라이브러리
# CartPole 등 다양한 시뮬레이션 환경 제공
!pip install gymnasium -q

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn

# TensorDataset: (입력, 정답) 쌍을 묶어주는 클래스
# DataLoader: 데이터를 배치 단위로 잘라서 공급 (MNIST에서 사용한 것과 동일)
from torch.utils.data import DataLoader, TensorDataset

# -----------------------------------------------
# 2. 전문가 정책 정의
# -----------------------------------------------

def expert_policy(obs):
    """
    규칙 기반 전문가 에이전트
    obs[2]: 막대 각도
    obs[3]: 막대 각속도
    각도만 보면 이미 늦을 수 있어서 각속도도 함께 고려
    """
    pole_angle    = obs[2]  # 막대가 얼마나 기울었는지
    pole_velocity = obs[3]  # 막대가 얼마나 빠르게 기울고 있는지

    if pole_angle + 0.1 * pole_velocity > 0:
        return 1  # 오른쪽으로 기울고 있음 → 오른쪽 이동
    else:
        return 0  # 왼쪽으로 기울고 있음 → 왼쪽 이동

# -----------------------------------------------
# 3. 전문가 데이터 수집 (cartpole_expert_data.ipynb와 동일)
# -----------------------------------------------

# Behavior Cloning의 핵심:
# 전문가가 각 상황(state)에서 어떤 행동(action)을 했는지 기록
# 나중에 모델이 "이 state → 이 action" 패턴을 학습

env = gym.make("CartPole-v1")
states  = []  # 각 스텝의 환경 상태 저장
actions = []  # 각 스텝의 전문가 행동 저장

for episode in range(50):  # 50번 플레이
    obs, info = env.reset()

    for step in range(200):  # 최대 200 스텝
        states.append(obs)              # 현재 상태 저장
        action = expert_policy(obs)     # 전문가 행동 결정
        actions.append(action)          # 행동 저장

        obs, reward, done, truncated, info = env.step(action)

        if done or truncated:
            break

env.close()

# -----------------------------------------------
# 4. numpy → PyTorch 텐서 변환
# -----------------------------------------------

# FloatTensor: 실수형 텐서 (state는 연속적인 값)
# LongTensor:  정수형 텐서 (action은 0 또는 1, CrossEntropyLoss에 필요)
states_tensor  = torch.FloatTensor(np.array(states))
actions_tensor = torch.LongTensor(np.array(actions))

# -----------------------------------------------
# 5. DataLoader 설정
# -----------------------------------------------

# TensorDataset: states와 actions를 (state, action) 쌍으로 묶음
# DataLoader:    배치 단위로 데이터 공급 - 배치(Batch)란 전체 데이터를 나눠서 학습하는 단위
#   batch_size=64: 한 번에 64개씩 학습
#   shuffle=True:  매 epoch마다 순서 섞기
dataset    = TensorDataset(states_tensor, actions_tensor)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

print(f"states_tensor shape:  {states_tensor.shape}")
# → (10000, 4): 10000개 스텝, 각 스텝마다 관찰값 4개

print(f"actions_tensor shape: {actions_tensor.shape}")
# → (10000,): 10000개 스텝, 각 스텝마다 행동 1개

print(f"배치 수: {len(dataloader)}")
# → 10000 / 64 ≈ 157 배치

states_tensor shape:  torch.Size([10000, 4])
actions_tensor shape: torch.Size([10000])
배치 수: 157


### 전문가 데이터 → PyTorch 텐서 변환 결과

- **`states_tensor`**: `(10000, 4)`
  - 50 에피소드 × 200 스텝 = 10,000개 스텝
  - 전문가가 매 에피소드마다 200 스텝을 모두 버텼음을 의미
  - 각 스텝마다 관찰값 4개 (카트 위치, 카트 속도, 막대 각도, 막대 각속도)

- **`actions_tensor`**: `(10000,)`
  - 각 스텝에서 전문가가 선택한 행동 (0: 왼쪽, 1: 오른쪽)
  - LongTensor를 쓰는 이유: 다음 단계의 CrossEntropyLoss가 정수형(Long)을 요구

- **`배치 수`**: `157`
  - 10,000 ÷ 64 = 156.25 → 157배치
  - 마지막 배치는 64개보다 적을 수 있음

### MNIST와의 비교

| | MNIST | Behavior Cloning |
|:---|:---|:---|
| **입력** | 이미지 (784 픽셀) | 상태 (4개 관찰값) |
| **정답** | 숫자 (0~9) | 행동 (0 또는 1) |
| **데이터 수** | 60,000개 | 10,000개 |
| **배치 수** | 937 | 157 |

구조는 완전히 동일. 입력과 정답의 형태만 다를 뿐.

#-------------
#모델 정의
#-------------

In [2]:
# -----------------------------------------------
# BC 모델 정의
# -----------------------------------------------

class BCPolicy(nn.Module):
    """
    Behavior Cloning 정책 모델

    MNIST 분류기와 구조가 거의 동일함:
      MNIST:  이미지(784) → 숫자(10)   ← 784개 픽셀을 보고 숫자 분류
      BC:     상태(4)     → 행동(2)    ← 4개 관찰값을 보고 행동 분류

    둘 다 결국 "입력을 보고 어떤 클래스인지 맞히는" 분류 문제
    """
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(

            # 입력층: 관찰값 4개 → 64개 뉴런
            # 4개의 숫자(카트 위치, 카트 속도, 막대 각도, 막대 각속도)를
            # 64개의 특징으로 변환
            nn.Linear(4, 64),

            # 활성화 함수: 비선형성 추가
            # 없으면 Linear를 아무리 쌓아도 선형 모델과 동일
            nn.ReLU(),

            # 은닉층: 64개 → 64개
            # 더 복잡한 패턴을 학습하기 위한 추가 레이어
            nn.Linear(64, 64),
            nn.ReLU(),

            # 출력층: 64개 → 행동 2개
            # 출력값: [왼쪽 점수, 오른쪽 점수]
            # 점수가 높은 쪽이 예측 행동
            # CrossEntropyLoss가 내부적으로 Softmax를 포함하므로
            # 여기서는 Softmax를 따로 붙이지 않음
            nn.Linear(64, 2)
        )

    def forward(self, x):
        # 입력 x가 모델을 통과하면서 순전파 진행
        # x shape: (배치 크기, 4) → 출력 shape: (배치 크기, 2)
        return self.model(x)

# -----------------------------------------------
# 모델 생성
# -----------------------------------------------

model = BCPolicy()

# -----------------------------------------------
# 손실함수 설정
# -----------------------------------------------

# CrossEntropyLoss: 분류 문제에 사용
# 내부적으로 Softmax + NLLLoss를 합친 것
# 정답 레이블로 LongTensor(정수형)를 요구
criterion = nn.CrossEntropyLoss()

# -----------------------------------------------
# 옵티마이저 설정
# -----------------------------------------------

# Adam: 각 파라미터마다 lr을 자동으로 조절
# lr=0.001: Adam의 기본값, 대부분의 경우 잘 동작
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# -----------------------------------------------
# 모델 확인
# -----------------------------------------------

# 모델 구조 출력
print(model)

# 전체 파라미터 수 계산
# p.numel(): 텐서의 전체 원소 수
total = sum(p.numel() for p in model.parameters())
print(f"\n전체 파라미터 수: {total:,}개")

# 참고: MNIST 모델은 109,386개
# BC 모델은 입력이 784→4로 줄었으므로 훨씬 적음

BCPolicy(
  (model): Sequential(
    (0): Linear(in_features=4, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=2, bias=True)
  )
)

전체 파라미터 수: 4,610개


#-------------
#학습루프
#-------------

In [3]:
epochs = 10  # 전체 데이터를 10번 반복 학습
             # MNIST(5번)보다 많은 이유:
             # 데이터가 10,000개로 적어서 더 많이 반복해야 함

for epoch in range(epochs):
    model.train()       # 학습 모드
    running_loss = 0.0

    for states_batch, actions_batch in dataloader:
        # states_batch.shape:  (64, 4) ← 64개의 관찰값
        # actions_batch.shape: (64,)   ← 64개의 정답 행동

        # 1. 순전파: 모델이 행동 예측
        # outputs.shape: (64, 2) ← 왼쪽/오른쪽 점수
        outputs = model(states_batch)

        # 2. Loss 계산
        # 예측한 점수와 정답 행동을 비교
        loss = criterion(outputs, actions_batch)

        # 3. 기울기 초기화
        # 이전 배치의 기울기가 누적되지 않도록
        optimizer.zero_grad()

        # 4. 역전파
        # 각 파라미터의 기울기 계산
        loss.backward()

        # 5. 파라미터 업데이트
        # 기울기 방향으로 weight, bias 조정
        optimizer.step()

        running_loss += loss.item()

    # epoch당 평균 Loss 출력
    avg_loss = running_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

print("\n학습 완료!")

Epoch 1/10, Loss: 0.4702
Epoch 2/10, Loss: 0.3496
Epoch 3/10, Loss: 0.3383
Epoch 4/10, Loss: 0.3285
Epoch 5/10, Loss: 0.3143
Epoch 6/10, Loss: 0.2942
Epoch 7/10, Loss: 0.2679
Epoch 8/10, Loss: 0.2378
Epoch 9/10, Loss: 0.2071
Epoch 10/10, Loss: 0.1781

학습 완료!


# -----------------------------------------------
# BC 모델 성능 평가
# -----------------------------------------------

In [4]:


model.eval()  # 평가 모드 (Dropout 등 비활성화)

correct = 0
total = 0

with torch.no_grad():  # 기울기 계산 불필요 → 메모리 절약
    for states_batch, actions_batch in dataloader:

        # 모델 예측
        outputs = model(states_batch)
        # outputs.shape: (64, 2) ← 왼쪽/오른쪽 점수

        # 점수가 높은 쪽 = 예측 행동
        # dim=1: 열 방향(클래스 방향)에서 최댓값 찾기
        _, predicted = torch.max(outputs, dim=1)

        total   += actions_batch.size(0)
        correct += (predicted == actions_batch).sum().item()

accuracy = correct / total * 100
print(f"훈련 데이터 정확도: {accuracy:.2f}%")
print(f"맞힌 개수: {correct} / {total}")

# -----------------------------------------------
# 실제 CartPole 환경에서 테스트
# -----------------------------------------------

env = gym.make("CartPole-v1")
episode_rewards = []

for episode in range(10):  # 10번 플레이
    obs, info = env.reset()
    total_reward = 0

    for step in range(200):
        # 관찰값 → 텐서 변환
        obs_tensor = torch.FloatTensor(obs).unsqueeze(0)
        # unsqueeze(0): (4,) → (1, 4) ← 배치 차원 추가

        # 모델이 행동 예측
        with torch.no_grad():
            output = model(obs_tensor)
            action = torch.argmax(output).item()
            # argmax: 가장 높은 점수의 인덱스 = 예측 행동

        obs, reward, done, truncated, info = env.step(action)
        total_reward += reward

        if done or truncated:
            break

    episode_rewards.append(total_reward)
    print(f"Episode {episode+1}: {total_reward} 스텝 버팀")

env.close()

print(f"\n평균 스텝: {sum(episode_rewards)/len(episode_rewards):.1f}")
print(f"전문가 평균: 200.0 스텝")
print(f"랜덤 평균:   10~44 스텝")

훈련 데이터 정확도: 92.28%
맞힌 개수: 9228 / 10000
Episode 1: 200.0 스텝 버팀
Episode 2: 200.0 스텝 버팀
Episode 3: 200.0 스텝 버팀
Episode 4: 200.0 스텝 버팀
Episode 5: 200.0 스텝 버팀
Episode 6: 200.0 스텝 버팀
Episode 7: 200.0 스텝 버팀
Episode 8: 200.0 스텝 버팀
Episode 9: 200.0 스텝 버팀
Episode 10: 200.0 스텝 버팀

평균 스텝: 200.0
전문가 평균: 200.0 스텝
랜덤 평균:   10~44 스텝
